In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
AFF_PREFIX = 'oa-top'
TOP_AFF_RANK = 50
MATCHING_FIRST_YEAR = 2012
MATCHING_LAST_YEAR = 2016
USE_OPENALEX = True

ROOT_DIR = Path('/projects/ai_science_alphafold')
RAW_DIR = ROOT_DIR / 'raw'
OUT_DIR = ROOT_DIR / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print({'RAW_DIR': str(RAW_DIR), 'OUT_DIR': str(OUT_DIR)})

In [ ]:
if USE_OPENALEX:
    pub2unnested = pd.read_csv(RAW_DIR / 'pub2unnested.csv')
    pub2unnested = pub2unnested.loc[
        (pub2unnested['type'] == 'article')
        & (pub2unnested['referenced_works_count'] > 0)
        & (pub2unnested['publication_year'] >= 1800)
    ].copy()

    pub2year = pub2unnested[['id', 'publication_year']].rename(columns={'id': 'PaperID', 'publication_year': 'Year'})
    pub2year['PaperID'] = pub2year['PaperID'].astype(str).str.lstrip('W').astype('int64')

    pub2source = pd.read_csv(
        RAW_DIR / 'pub2source.csv',
        header=None,
        names=['PaperID', 'SourceID', 'oa', 'oaType', 'mainClass', 'Classes'],
        usecols=['PaperID', 'SourceID'],
        dtype='string',
    )
    pub2source['PaperID'] = pub2source['PaperID'].str.lstrip('W').astype('int64')
    pub2source['JournalID'] = pub2source['SourceID'].str.replace('https://openalex.org/', '', regex=False)
    pub2source = pub2source[['PaperID', 'JournalID']]

    pub2year = pub2year.merge(pub2source, on='PaperID', how='left')
else:
    pub2year = pd.read_csv(RAW_DIR / 'Papers.tsv', sep='\t', usecols=['PaperID', 'JournalID', 'Year'])
    pub2year = pub2year.loc[pub2year['JournalID'].notna()].copy()
    pub2year[['PaperID', 'JournalID', 'Year']] = pub2year[['PaperID', 'JournalID', 'Year']].astype(
        {'PaperID': 'int64', 'Year': 'int64'}
    )

pretrend_pub2year = pub2year.loc[pub2year['Year'].between(MATCHING_FIRST_YEAR, MATCHING_LAST_YEAR)].copy()

print(pub2year.shape, pretrend_pub2year.shape)

In [ ]:
bio_med_children = pd.read_csv(
    RAW_DIR / 'FoS.txt', sep='\t', header=None, names=['FieldOfStudyId', 'ChildFieldOfStudyId']
)
bio_med_domain = (
    bio_med_children.loc[bio_med_children['FieldOfStudyId'].isin([86803240, 71924100]), 'ChildFieldOfStudyId']
    .dropna()
    .astype('int64')
)

pub2fos = pd.read_csv(RAW_DIR / 'PaperFields.tsv', sep='\t', usecols=['PaperID', 'FieldID'])
pub2fos = pub2fos.loc[pub2fos['FieldID'].isin(set(bio_med_domain))].copy()
pub2fos[['PaperID', 'FieldID']] = pub2fos[['PaperID', 'FieldID']].astype('int64')

print(pub2fos.shape)

In [ ]:
pub2aff = pd.read_csv(
    RAW_DIR / 'pub2corresponding.csv',
    header=None,
    names=['oaid', 'author_id', 'author_order', 'is_corresponding', 'affs_id'],
    dtype={
        'oaid': 'string',
        'author_id': 'string',
        'author_order': 'Int64',
        'is_corresponding': 'boolean',
        'affs_id': 'string',
    },
)
pub2aff['affs_id'] = pub2aff['affs_id'].fillna('').map(lambda s: [x for x in s.split(';') if x])

team_size = pub2aff.groupby('oaid', dropna=False)['author_order'].transform('max')
pub2aff_main = pub2aff.loc[pub2aff['author_order'].eq(team_size)].copy()
pub2aff_main = pub2aff_main.drop(columns=['author_order'])
pub2aff_main['author_order'] = team_size.loc[pub2aff_main.index].astype('Int64')

print(pub2aff.shape, pub2aff_main.shape)

In [ ]:
pub2ids = pd.read_csv(
    RAW_DIR / 'pub2ids.csv',
    header=None,
    names=['oaid', 'pmid', 'magid'],
    dtype={'oaid': 'string', 'pmid': 'string', 'magid': 'Int64'},
)
pub2ids = pub2ids.loc[pub2ids['pmid'].notna()].copy()

match AFF_PREFIX:
    case 'oa-top':
        aff2rank = pd.read_csv(
            RAW_DIR / 'institution2rank.csv',
            header=None,
            names=['institution_id', 'display_name', 'type', '2yr_i10_index', '2yr_h_index'],
        )
        aff2rank = aff2rank.loc[aff2rank['type'].eq('education')]
        aff2rank = aff2rank.sort_values('2yr_h_index').tail(TOP_AFF_RANK).copy()
    case _:
        raise ValueError(f'Unsupported AFF_PREFIX for pandas rewrite: {AFF_PREFIX}')

assert aff2rank['institution_id'].dropna().nunique() == TOP_AFF_RANK
print(pub2ids.shape, aff2rank.shape)

In [ ]:
pretrend_pub2year.to_csv(OUT_DIR / 'pretrend_pub2year.csv', index=False)
pub2year.to_csv(OUT_DIR / 'pub2year.csv', index=False)
pub2fos.to_csv(OUT_DIR / 'pub2fos.csv', index=False)
pub2aff.to_csv(OUT_DIR / 'pub2aff.csv', index=False)
pub2aff_main.to_csv(OUT_DIR / 'pub2aff_main.csv', index=False)
pub2ids.to_csv(OUT_DIR / 'pub2ids.csv', index=False)
aff2rank.to_csv(OUT_DIR / 'aff2rank.csv', index=False)

run_config = pd.DataFrame(
    [
        {
            'AFF_PREFIX': AFF_PREFIX,
            'TOP_AFF_RANK': TOP_AFF_RANK,
            'MATCHING_FIRST_YEAR': MATCHING_FIRST_YEAR,
            'MATCHING_LAST_YEAR': MATCHING_LAST_YEAR,
            'USE_OPENALEX': USE_OPENALEX,
        }
    ]
)
run_config.to_json(OUT_DIR / 'run_config.json', orient='records', indent=2)

print('Wrote base tables to', OUT_DIR)